# Reinforcement learning and pre-training

Over the past week or so we've been talking about the basic structure of machine learning algorithms. In many ways, they are just "fancier" versions of the modeling techniques we covered in previous units. My plan for this notebook it to introduce some if the ideas and techniques that sit behind modern large language models (LLM) that power chat bots and coding agents, like the ones involved in the Hugging Face incident.

We saw in the last notebook that the data used has a **huge** impact on the model and its validity. This is in someway the old computer science addage "garbage in, garbage out." If your training data is the *entire* internet and every book or paper ever written (a lot of that is just kinda garbage) this begs a question: How can you take that mess of data and build a model than can solve previously unsolved [research-level math problems](https://www.bbc.com/news/articles/cy7zygy3rl2o)? 

The answer (or at least parts of it) live in **Pre-training** and **Reinforcement learning.** 

When we say *pre-training* we mean doing a fully automated task to try to learn something about the data. In the example of an LLM, that might be just trying to predict the next word given the previous text in a document. In order to get good at this skill of prediction, you need to learn something about how language is constructed. It also does not need a human or well-curated dataset. It can predict the next word in a tweet just as easily as the next word in a Tolstoy novel. 

The name *reinforcement learning* comes from psychology. It's how we, as mammals, learn. The basic ideas is that when the model does something *good* we give it a virtual award (some points). When a model does something *bad* we give it a punishment by removing points. We then give a model a bunch of tasks, and score the outcome, incentivising the model to get as high a score as possible. 

Many of the recent gains in the performance of LLMs have been do in reinforcement learning, and it is part of the reason why these models are so good at math and coding. Because math and coding problems often have *verifiable answers* (it's easy to say if the answer is correct or not) you can run a reinformed learning loop on these sorts of problems easily. Other problems, like the answer to a chatbot message are squishier. This is part of the issue with sycophancy in chatbots, if part of the training is RL on human input, and humans liked to be told they are good and smart, is it really a surprise we see these chatbots behaving like they have? 

### Setting up the code 

I've decided to hide most of the code from you, and only expose a few functions. The entirety of the code can be found [here](), most of it was written by one of those coding agents I mentioned above. I have also trained a few different version of our test model the same way the authors of that New York Times piece did in the interest of time.  

Run the cell below to import all of the python packages we will use in this notebook, including the custom maze code.

In [ ]:
# Install the maze code (takes about a minute)
%pip install -q git+https://github.com/anthoak13/HNR-MazeSolver@maze-rl

#%pip install -q --force-reinstall --no-deps git+https://github.com/anthoak13/HNR-MazeSolver@maze-rl
# Import the maze code and other libraries used in this notebook
import mazerl as mz
import pandas as pd
from IPython.display import display

mz.check_gpu()   # warns you if this runtime has no GPU

def compare(results):
    return pd.DataFrame(results).T
def printRewards(r):
    return pd.DataFrame({"points": [r.flag, r.outside, r.probe, r.new_cell, r.step]},
                        index=["reach the flag (ends the attempt)",
                               "each move that ends outside the maze",
                               "probing a wall (weak spots give way, solid walls don't)",
                               "stand on a square for the first time",
                               "every move, no matter what"])


Run the cell below to make sure the code loaded correctly. You should see the images one of the mazes from the NYTimes article when you do.

In [ ]:
# Maze where the flag is behind a weak spot in the wall"
display(mz.show_maze(mz.scenario("enclosed")))


## Defining the model

Our testbed for this exploration will be a maze-solving agent. In this model, there are 14X14 mazes sitting in a 18X18 world. Each maze has a flag sitting somewhere in the middle. The agent is a dot who can walk around the maze to try to find the flag. Some of the walls are "vulnerable" and if the agent probes them, can discover a hidden passageway through the maze. As you can see above, I have visualized these will a light gray square.

Our eventual goal is to train an agent who can solve any random maze given to them, even a very hard one like this "enclosed" example. The code cell below will pull a working version of the model I trained based on the (limited) information for the Times authors. It then runs on the maze above. You can have it solve a random maze by changing which line of code is uncommented.

In [ ]:
# The full-scale agent with the default rewards
base = mz.load_pretrained("base")

maze = mz.scenario("enclosed")
maze = mz.random_mazes(1)[0]

mz.playback(base, maze=maze, n_agents=5, seed=0)



We're going to take this in four parts:
1. Defining the data and behavior for the agent
2. Pre-training the agent to learn about mazes
3. Training the agent to solve mazes using RL (and see how they can learn to solve mazes not in their training data)
4. Examining how changes in rewards/punishments can effect the behavior of our agents
5. Emergent behavior of agents

## Part 1: Defining an agent

For our agent to be an effective maze solver, it will need some skills (but not too many, it wouldn't be much of a challange if our agent could see the intire maze now would it!).

**What skills or information (i.e., data) would you give the agent if you were building it from scratch?**

- 
- 

----

### What an agent sees

For agent to have any hope of success, we will need to give it three things. A memory about the size of the maze, some basic vision, and a vague sense for what direction the flag might be in. In this model the agent has:
1. An 11X11 sized memory centered on itself. 
2. The ability to see up to two squares away, but its vision is blocked by solid walls. 
3. A memory of where the agent has been before
4. A memory of what walls the agent has prodded before
5. A single number (1-8) that indicates in what direction the flag is in

For the memory, there 9 things which could be true we need to track:
- not seen
- open floor
- outside
- wall
- flag
- map edge
- can see now
- have prodded
- have stood

For each cell, these can either be true (1) or false (0) so what we do is just store nine $11\times 11$ lists where each cell is either a 1 or a 0 depending on the agent's knowledge. The direction has eight options so we pass eight numbers (all zero but the correct direction which i 1), so in the end we are passing $9\times 11\times 11 +8 = 1097$ numbers to the model (or 1097).

If you run the cell below, it will show what an agent would see. You can play around with different examples by changing the step (larger number means the agent has wandered for longer) and seed (a different "trial"). 

In [ ]:
# One new maze, used throughout Part 1
maze = mz.random_mazes(1)[0]

mz.show_observation(maze=maze, step=10, seed=0)


### How an agent behaves

The behavior of our agent will be determined by the **output** of our model. When we were identifying digits, the output of our model was 10 numbers all 0-1. We said the digit the model identified was whichever was largest. We will do something here where the output of our model will be 4 numbers that correspond to moving: up, down, right, and left. Technically, there is a fifth output but that is just used for training. It is sort of like the model's "confidence" in its ability to solve the maze.

Unlike the digits, where we always picked the most likely outcome, when we are using the model we will treat the outputs as a probability. For example, if the output was `[0.5, 0.5 , 0, 1]` we would interpret that as a 25\% chance of going up,  a 25\% chance of going down, and a 50\% chance of going left.

Once the agent moves, we can update our memory and pass it all back to the model to generate a new prediction for our next move. And then repeat until we reach the flag (hopefully)

You can see this in the example below where each of the five agents (all using the same model) take different paths. Some take the 25\% chance and some take the 50\% chance.

In [ ]:
base = mz.load_pretrained("base")
maze = mz.scenario("enclosed")
mz.playback(base, maze=maze, n_agents=5, seed=0)

### How an agent thinks

Now that we have defined the input and output, we can discuss what the "brains," or network architecture looks like. I made a best guess for what this would have been in the NYTimes article based on the limited information. The ML network has two halves:

The first half is a "vision" or "perception" half. It's responsible for taking in all our memory (as well as the eight compass numbers) and distilling it into 100 "features" which describe the world around the agent. 

The second half takes as its input the 100 "features" from the vision section and makes a prediction. This is the "reasoning" part. It has one more fully connected layer before the output. This is like the last hidden layer on our character finder, but now the hidden layers are 100 neurons rather than 16. The final output layer is 4 numbers, referring to the four directions.

## Part 2: (Pre)-training an agent

In principle, there are a bunch of different tasks we could do with a maze-based agent. It could search for a flag or it could try to break out of a maze. Both of these need different strategies, but they share a lot of common knowledge. An agent needs to learn about walls, and vulnerable sections of wall, and how rooms are constructed, etc.

Just like in LLMs, we can "pre-train" a model by having it ingest the entire internet and having it guess the next token, we can do something similar for our maze model. For a task to be a good pre-training task it should not need any human input. In our case, we will have the agent **predict** what each square in its memory is, that it has not seen yet. In order to be good at this task, you need to understand something about how mazes are constructed which should be useful in solving all sorts of maze-based task!

### The data

In order to get an agent to predict what is within the "fog of war" in its memory, we will need some basic agent that can walk around a maze. This is something we can do easily without ML. I wrote a little script that has the agent walk towards the nearest empty square it has seen but hasn't stood on yet (most of the time, there is a 30\% chance it just moves randomly). We let the agent wander through randomly chosen training mazes (up to 200 steps in each) until we had 150,000 snapshots. After each step we save a snapshot of the current memory of the agent (the nine $11\times 11$ maps discussed above). We also save the "true" state of each cell in the memory block, even if the agent hasn't seen it yet.

Critically, everything we just described was with NO ML. All we have done is automatically generated a dataset to use for pretraining. The cell below shows the path the explorer might have taken of a random maze.




In [ ]:
# The scripted explorer wandering one maze
maze = mz.random_mazes(1)[0]
mz.playback("explorer", maze=maze, n_agents=1)

### The task

For each "snapshot" we took, we ask the model to answer the question **for each square it hasn't seen yet, what's really
there?** 1) Open floor, 2) outside, 3) wall, or 4) the edge of the map. 

Mechanically, this means we are taking the 100 "features" from the vision model and generating 4 output numbers for each of the 121 cells. This is exactly the same as our character recognition, except we have 4 answers (instead of 10) and we are doing it once for each cell, rather than once for the whole image.

**If the model were randomly guessing (or always guessing the most common answer), how often would it get the hidden cells correct?**

Let's take a look at how the model does before and after this round of pre-training. If you run the cell below, it will generate two images showing the guesses of the model before and after pre-training for the same step of the same maze shown above. 

In [ ]:
# Load the network that was pretrained this way (no rewards involved)
untrained = mz.untrained_perception()
pretrained = mz.load_perception("perception")

# Optional: run the pretraining yourself instead of loading it (about 20 seconds on a GPU or ~10 minutes on a CPU)
#pretrained = mz.pretrain_perception()


display(mz.show_pretraining_quiz(untrained, maze=maze, step=30, seed=0))
display(mz.show_pretraining_quiz(pretrained, maze=maze, step=30, seed=0))

If we look at how this does across all snapshots in the training data, we can see that we are doing much better than random!

In [ ]:
t_un = untrained.test
t = pretrained.test
print(f"Hidden squares guessed right before training: {t_un['map_acc_unseen']:.0%}\n"
      f"Hidden squares guessed right after training: {t['map_acc_unseen']:.0%}\n"
      f"   (always guessing the most common kind: {t['map_acc_majority_baseline']:.0%})")

## Part 3: Finding a flag with RL

### Defining a goal

Pretraining gave our network general knowledge about mazes, but it doesn't *want* anything yet. To train the model to perform a specific task, we will need to decide how to structure its rewards and punishments. There are a number of actions that the agent can take that we can reward or punish. We should reward reaching the flag and finding a new tile to stand on. We will also punish ending up outside the maze or probing a wall, since these might be "undesirable" traits for the model to learn.

In [ ]:
# The default points for each event (negative = penalty)
r = mz.Rewards()
printRewards(r)


The flag is worth far more than anything else, so reaching it is the agent's main goal. The small penalties make wasted moves cost something, and the small reward for new squares encourages exploring.

An attempt's **score** is the total of all its points. In many of the visualizations above, you can see the agents scores along the bottom and little "pop-ups" as the agent loose/gain points. The table below shows where the points came from in one attempt on our maze, for an agent moving at random and for a fully trained agent. Re-running it will run a new agent so you will see the numbers change.

In [ ]:
compare({"moving at random": mz.score_breakdown(None, maze=maze, seed=0, plot=False),
         "fully trained": mz.score_breakdown(base, maze=maze, seed=0, plot=False)})

**Our goal is to get as many points as possible.** Ideally, that means solving the maze quickly. Now that we can define a score, how do we tell if we are making a good move or a bad move half way through solving the maze?

### Assessing the goal

We had it easy before with our digits and pre-training. We were either right or wrong. Now we have a large list of individual decisions that lead to a final score. Say it takes 80 moves to get to the end, were they all important? Probably not. The way we handle this is to let the agent run until it takes 200 steps or solves the maze, then we go back and assign a score for *each move.* The moves score is the sum of all the points the agent would get from then to the end of the maze.

**If we want the maze solved quickly, should a point earned from the next move be worth more of less than a point earned 100 moves from now?** Explain your reasoning.


Once you've written down your answer, run the code cell below to see how many points the agent gets for the flag, depending on how many moves are remaining.

In [ ]:
# What the +5 for the flag is worth from where the agent stands, if the flag is this many moves away
moves = [1, 10, 25, 50, 100, 200]
pd.DataFrame({"moves to the flag": moves,
              "the +5 is worth now": [round(5 * 0.99 ** m, 2) for m in moves]}).set_index("moves to the flag")

Earlier, I said there was a fifth output (besides the directions) used for training. That value is really the number of points the agents *expects* to get from this move. If it expected a lower number of points than the scorer gave it, that means this was a good move it should do more often. If it expected a higher number of points, that means this was a bad move. It's the difference between this prediction and reality that is used as the cost function we are optimizing against.

### Training: putting it all together

Now we can define our rewards, and run the training loop on our pre-trained network. We'll keep the pre-trained vision half frozen and only let RL change the
reasoning half: the last hidden layer and the outputs. That's about 10,000 of the 230,000 numbers in the network.

Run the cell below to train the model using the rewards from above. If you are on a CPU, not a T4, then uncomment the second line which loads a fully trained model.

In [ ]:
# trains the flag-finding on 20,000 attempts using the pre-trained network.
#flag_live = mz.train(perception=pretrained.agent, freeze_perception=True, episodes=20_000,
#                     checkpoint_at=(1_000, 2_000, 5_000, 10_000))

# This loads a model rather than training it
flag_live = mz.load_pretrained("flag_frozen_pretrained")

One of the benefits of pre-training is that RL doesn't have to learn everything from scratch. In fact, the version with random vision is just not really getting better at all. You can see that below, were of we trained for the same amount of time on a model without any pre-training the agents do much worse. In the animation below, each agent is at a different stage of training. The top row is with pre-training and the bottom without pre-training.

In [ ]:
# The same training on top of fixed RANDOM layers (trained ahead of time)
flag_random = mz.load_pretrained("flag_frozen_random")


# Watch the pretrained and random agents at several stages of its training
display(mz.training_grid(flag_live, n_mazes=3))
display(mz.training_grid(flag_random, n_mazes=3))

(Optional!) For those interested in seeing how pre-training effects the model behavior, run the cell below to plot some stats vs run number.

In [ ]:
mz.plot_training([flag_live, flag_random], labels=["pretrained layers", "random layers"])

Answer the following questions:

**Why do you think LLMs use RL and pre-training?**

**How do you think they define their rewards/punishments? Does it vary by task?**

# Part 4: Reward hacking

**Run the cell below to start training a baseline model while you read this section** (so actually read while it trains! It should take ~2.5 min). 



In [ ]:
# Train a model for 20,000 attempts with the default rewards
default_rewards = mz.Rewards() 
#default_run = mz.train(rewards=default_rewards, episodes=20_000, checkpoint_at=(1_000, 5_000, 10_000))

# This loads a model rather than training it
default_run = mz.load_pretrained("default_20k")


A danger in RL is how do we get the agent to do what we want? This is called *alignment* and even in a toy model like this one, it can get messy. Our goal, designing this model, is to reach the flag as quickly as possible.

Let's train two models on 20k escape attempts. This time we will train both the vision and reasoning systems together (they do better trained together than separately). The only thing we will change is our rewards and punishments. 

**What reward could we change so the agent takes longer to solve the maze (on average) while still solving it in most runs?**

Once you have made your prediction, modify the cell below to change that parameter and re-run the cell. Make sure what is in the printed table matches what you wanted (remember to uncomment the line you are changing!)

In [ ]:
# Create a new set of rewards that encourages exploration
                    
explore_rewards = mz.Rewards()
# Everything you can change (shown at the default values):
# explore_rewards.flag = 5.0               # reaching the flag (ends the attempt)
# explore_rewards.outside = -0.01          # each move that ends outside the maze
# explore_rewards.probe = -0.01            # walking into a wall (weak spots give way, solid walls don't)
# explore_rewards.new_cell = 0.005         # standing on a square for the first time
# explore_rewards.step = 0.0               # every move, no matter what

display(printRewards(default_rewards))
display(printRewards(explore_rewards))

Once you are happy, run the cell below to train your new model and print out some information about its performance over time!

In [ ]:
#explore_run = mz.train(rewards=explore_rewards, episodes=20_000, checkpoint_at=(1_000, 5_000, 10_000))

# This loads a model rather than training it ith new_cell = 0.05
explore_run = mz.load_pretrained("explore_20k")

mz.plot_training([default_run, explore_run], labels=["default", "Changed rewards for exploration"])

**Based on the plots above, do you see evidence of reward hacking?** In other words, is your new agent taking longer to reach the flag? Is it still reaching the flag just as often eventually?

Let's look at a few agents from the training run. **Do the animations agree with what you inferred from the plots above? Why or why not?**

In [ ]:
test_maze = mz.random_mazes(1, seed=3, kind="open")[0]
display(mz.playback(default_run, maze=test_maze, n_agents=5, seed=0))
display(mz.playback(explore_run, maze=test_maze, n_agents=5, seed=0))

A common outcome, although it depends on the details of what you changed is that the second agent tours the outside of the maze collecting points for new squares before it heads for the flag. Its score goes up while it gets worse at the actual job. That's **reward hacking**. It's doing exactly what we incentivized it to do, not what we actually meant. This is a small scale version of the sycophancy of chatbots that we commonly see.

To be a little more quantitative, the cell below calculates and compares some averages over 300 mazes.   

In [ ]:
new_mazes = mz.random_mazes(300, seed=4)
compare({"default": mz.evaluate(default_run, mazes=new_mazes),
         "Changed rewards for exploration": mz.evaluate(explore_run, mazes=new_mazes)})[
    ["avg score", "reached flag", "moves to flag", "avg moves outside"]]

## Part 5: Emergent behavior

Reward hacking is distinct from the **emergent behavior** that can arise from models trained using RL. To tie it back to our discussion in class today, reward hacking is related to sycophancy in chatbots, while this idea of emergent behavior is more similar to the hugging face incidents. 

In the Hugging Face hacks, the agents combined numerous skills in new ways to escape the lab and hack another company. All in service of maximizing their score (apparently, there was not sufficient punishment for committing hacks that would be felonies if done by a person!)

My goal here is for you to see how this can arise in our maze model. The arguments here follow directly from those in the reading for today where the authors of that piece centered one maze in particular:

In [ ]:
mz.show_maze(mz.scenario("enclosed"))

This is an *extremely* hard maze. To get to the flag the agent needs to walk around, then break in, then break through another wall. While being trained, the agetns never saw a maze like this. But they did see three different types of mazes that involve one of these skills in isolation. 

In [ ]:
# One example of each kind of maze the agents trained on (equal numbers of each)
kinds = {"open":           "Open: there is a route to the flag inside the maze",
         "interior_break": "Weak spot: the only way to the flag is through a weak wall (and an internal path)",
         "outside_only":   "Outside: the only way is out the entrance and back in another gap (but no weak walls)"}
mz.player.show([{**mz.random_mazes(1, seed=0, kind=k)[0].to_dict(), "agents": [], "title": t}
                for k, t in kinds.items()])

What agents can learn to do is *combine* these skills in new ways. That's what's required to solve the "hard maze" and is, in some way, analogous to AI agents "going rogue."

Let's consider the agent you just trained using 20,000 attempts. If we let an agent take 500 attempts, how many do you think will make it though each stage (leave the maze, then break in, then break into the flag room)?

**Make your prediction before running the cell below.** It shows both some statistics from your trained model, and some attempts by the agent.

In [ ]:
display(mz.playback(default_run, maze="enclosed", n_agents=10))
display(mz.breakout_stages(default_run))

As you can see, it really is not that good at this hard maze yet. It breaks in a lot, but rarely makes it to the flag.

But does that mean it can't be done, or our agent just isn't smart enough? In the article, they trained their model for 126,000 attempts. So let's do the same. Since it takes while to train this, I've provided the already trained model for you.

In [ ]:
full_run = mz.load_pretrained("base")
display(mz.playback(full_run, maze="enclosed", n_agents=10, seed=0))


**Based on the visualization, how often do you think the agent makes it through the three steps?** Once you have written your prediction run the code cell below.

In [ ]:
display(mz.breakout_stages(full_run))

**How do the probabilities of combining skills compare between the 20k and 126k training runs?** 

## Wrapping up

At the start of this notebook I asked how you can take a giant pile of (mostly garbage) data and end up with a model that can do something impressive. In our little maze world, we saw the two big pieces of the answer.

**Pre-training** let the model learn a ton about mazes without anyone labeling anything, just by guessing what was hiding in the fog of war. That general knowledge is what made everything else possible. With it, RL only had to change a tiny part of the network.

**Reinforcement learning** is what gave the model a goal. But the agent doesn't learn what we *meant*, it learns what we *rewarded*. If we made exploring worth a bit more, it happily toured the outside of the maze racking up points. And when we trained long enough, it combined skills it had learned separately to do something nobody ever taught it: leave the maze and break back in. It did what it was told, it found the route that scored the most points in both cases. The classic example of this is the paperclip maximizer (I'd highly recommend checking out the game [here](https://www.decisionproblem.com/paperclips/) if you haven't before.)

That's the same story as the chatbots that tell you how smart you are, and the agents in the Hugging Face incident. Nobody programmed those behaviors in. They fell out of the rewards and the training. And our agent is a 230,000-number network solving toy mazes. The models in the news are about a million times bigger, doing far messier tasks, with rewards that are much harder to get right.

**If you were training a chatbot, what reward would you be most worried about it hacking?**

## Bonus: Escaping the maze

Back in Part 2 I said a maze agent could do more than one job. It could look for a flag, or it could try to escape. Pre-training didn't care which; it just learned how mazes work. Let's put that to the test.

We'll take the exact same pre-trained vision half and give it a different goal: start where the flag would be, and get *out* of the maze. Stepping onto any square outside wins the +5. There's no compass this time (there's no flag to point to), but all the other rewards are the same.

This section stands on its own, so you only need to have run the setup cell at the top.

In [ ]:
# Load the pre-trained vision half (no rewards were involved in making it)
pretrained = mz.load_perception("perception")

# What the agent sees in the escape task (check the compass!)
mz.show_observation(task="escape", step=10, seed=0)

Now let's train it. Just like in Part 3, only the top of the network gets to change and the pre-trained vision stays frozen (about 2 minutes on a T4).

In [ ]:
# Train an escape agent for 20,000 attempts on top of the frozen pre-trained vision
#escape_run = mz.train(task="escape", perception=pretrained.agent, freeze_perception=True,
#                      episodes=20_000, checkpoint_at=(1_000, 2_000, 5_000, 10_000))

# This loads a model rather than training it
escape_run = mz.load_pretrained("escape_frozen_pretrained")

**Before you watch: what do you think this agent will do differently from the flag-finder in Part 3?**

In [ ]:
display(mz.playback(escape_run, maze=mz.random_mazes(1)[0], n_agents=5, seed=0))

# The same agent at different stages of its training
display(mz.training_grid(escape_run, n_mazes=3))

Same pre-trained network, different reward, completely different agent. This is how the big AI companies work too: one huge pre-trained model gets turned into a coding assistant, a chatbot, or a math solver depending on how it's rewarded afterward.